# Airbnb Italy – Estimated Revenue Prediction (v2)

One notebook that **drives the `airbnb_profit` package** – every number here is produced by the same cached,
scripted pipeline as `python scripts/run_pipeline.py`:

`prepare → split → tune (train only) → evaluate (test, once) → report`

**What this fixes versus the 3 original notebooks:** (1) test-set leakage in stacking/Bayes/GA,
(2) a target built from its own inputs + whole-data min-max scaling, (3) preprocessing learned before the split –
plus MAPE, train-score model selection, host leakage, GA bugs, string rank-correlations and non-reproducible setup
(see `README.md`).

## 0. Setup & configuration

In [ ]:
import sys, os, json, logging, warnings
from pathlib import Path

ROOT = Path.cwd().resolve()
if not (ROOT / "config").exists():           # notebook lives in notebooks/
    ROOT = ROOT.parent
sys.path.insert(0, str(ROOT / "src"))        # works with or without `pip install -e .`

import numpy as np, pandas as pd
import matplotlib.pyplot as plt
from IPython.display import display

from airbnb_profit import pipeline as P, features as F, target as T, evaluate as ev, viz
from airbnb_profit.config import load_config, with_overrides, resolve, config_hash

warnings.filterwarnings("ignore", category=FutureWarning)
logging.basicConfig(level=logging.INFO, format="%(asctime)s %(message)s", datefmt="%H:%M:%S")
pd.set_option("display.max_columns", 50, "display.width", 160, "display.float_format", "{:.3f}".format)

In [ ]:
# ---- user switches -----------------------------------------------------------------
QUICK = os.environ.get("AIRBNB_QUICK", "0") == "1"   # tiny budget for CI / smoke runs
FORCE = False                                        # True = ignore cached stage outputs
CITIES = None                                        # e.g. ["roma", "milano"]; None = all in config

cfg = load_config(ROOT / "config" / "default.yaml")
real_csv = resolve(cfg, "raw_csv")
if not real_csv.exists():
    print("!" * 78)
    print(f"{real_csv.name} not found -> falling back to SYNTHETIC data (numbers are NOT about Italy).")
    print("Place the real file in data/raw/ (see scripts/download_data.py) and re-run.")
    print("!" * 78)
    cfg = with_overrides(cfg, **{"data.source": "synthetic", "project.run_name": "demo_synthetic"})
if QUICK:
    cfg = with_overrides(cfg, **{"project.run_name": cfg["project"]["run_name"] + "_quick",
                                 "data.cities": ["bergamo", "roma", "firenze"], "data.synthetic_rows_per_city": 500,
                                 "tuning.n_trials": 4, "tuning.cv_folds": 3, "tuning.ga_population": 2,
                                 "stacking.cv_folds": 3, "evaluation.bootstrap_iterations": 30,
                                 "evaluation.permutation_repeats": 2})
print("run:", cfg["project"]["run_name"], "| config hash:", config_hash(cfg), "| source:", cfg["data"]["source"])
run = P.Run(cfg, force=FORCE)
CITIES = CITIES or run.cities

## 1. Prepare: target definition and data audit

**Target** (documented, row-wise, no dataset statistics):

`booked_nights = min(reviews_ltm / review_rate × avg_stay, max_occupancy × 365)`,  `revenue = price × booked_nights`,
model target = `log1p(revenue)`.

`price`, review and availability columns build the target and are **never features** – the pipeline asserts this.
This is *revenue*, not profit (no cost data) and the population is listings active in the last 12 months.

In [ ]:
audit = P.stage_prepare(run, CITIES)
display(audit)
display(pd.Series(cfg["target"], name="target parameters").to_frame().T)

In [ ]:
frames = {c: P.load_city(run, c) for c in CITIES}
viz.plot_target_distributions(frames); plt.show()

summ = pd.DataFrame({c: f["est_annual_revenue"].describe(percentiles=[.1, .5, .9]) for c, f in frames.items()}).T
display(summ.round(0))

### 1b. Do cities really differ? (effect size, not just p-values)
With ~10⁴ rows per city a one-way ANOVA gives p≈0 for almost any difference (the v1 conclusion). We add the
non-parametric Kruskal–Wallis test and the **effect size η²** (share of variance explained by "city").

In [ ]:
from scipy import stats
groups = [f["target_log_revenue"].to_numpy() for f in frames.values()]
H, p = stats.kruskal(*groups)
allv = np.concatenate(groups)
eta2 = sum(len(g) * (g.mean() - allv.mean()) ** 2 for g in groups) / ((allv - allv.mean()) ** 2).sum()
print(f"Kruskal-Wallis H={H:.1f}, p={p:.2g};  eta^2 (variance explained by city) = {eta2:.3f}")

## 2. Leakage guards (executable proof, not a promise)

In [ ]:
print("Forbidden as features:", sorted(F.FORBIDDEN_FEATURES)[:6], "... (", len(F.FORBIDDEN_FEATURES), "total )")
for c, f in frames.items():
    F.assert_no_leakage([x for x in f.columns if x not in ("est_annual_revenue", "target_log_revenue")])
print("OK: no city frame contains a target ingredient or post-hoc demand column.\n")

for c in CITIES:
    P.stage_split(run, c)
    X_tr, y_tr, g_tr, X_te, y_te, g_te, _ = P.load_split(run, c)
    assert set(g_tr).isdisjoint(g_te)
print("OK: train and test never share a host in any city.")
print("Features used by every model:", len(F.FEATURE_COLUMNS))
display(pd.DataFrame({"column": F.FEATURE_COLUMNS,
        "kind": ["numeric"] * len(F.NUMERIC) + ["one-hot"] * len(F.CAT_LOW) + ["target-encoded (in-fold)"] * len(F.CAT_HIGH) + ["multi-hot, vocab from train"]}).T)

## 3. Tune – on the training set only
Host-grouped CV; objective = mean CV RMSE on the log target. `tpe` = Bayesian optimisation, `ga` = genetic
algorithm (NSGA-II). Every model family is tuned **per city** (v1 tuned only the meta-learner and got identical
parameters everywhere).

In [ ]:
tuning = {c: P.stage_tune(run, c) for c in CITIES}
rows = [{"city": c, "model": r["model"], "sampler": r["sampler"], "best_cv_rmse": r["best_cv_rmse"], "n_trials": len(r["trials"])}
        for c, rs in tuning.items() for r in rs]
cv_table = pd.DataFrame(rows)
display(cv_table.pivot_table(index="city", columns=["model", "sampler"], values="best_cv_rmse"))

In [ ]:
c0 = CITIES[0]
viz.plot_tuning_history(tuning[c0], c0); plt.show()

## 4. Evaluate – the test set is used exactly once
Models fitted on train with tuned parameters, plus the baselines `dummy` (median) and `ridge`. The **stack** uses
out-of-fold base predictions on train and a non-negative linear meta-learner. The "selected" model per city is chosen by
**train CV**, never by test score. CIs resample whole hosts.

In [ ]:
for c in CITIES:
    P.stage_evaluate(run, c)
rep = P.stage_report(run)
metrics = rep["metrics"]
cols = ["city", "model", "selected_by_cv", "r2_log", "r2_log_lo", "r2_log_hi", "rmse_log", "medae_eur", "wape", "spearman"]
display(metrics[cols].round(3).sort_values(["city", "rmse_log"]).head(40))

In [ ]:
display(rep["summary"])
viz.plot_model_comparison(metrics); plt.show()
viz.plot_metric_heatmap(metrics, "rmse_log"); plt.show()

### 4b. Is a model *significantly* better? (paired across cities)

In [ ]:
display(rep["tests"].round(4))
files = sorted(run.eval.glob("*_stack_weights.json"))
display(pd.DataFrame([json.loads(p.read_text()) for p in files], index=[p.name.split("_")[0] for p in files]).round(3))

In [ ]:
sel = metrics[metrics.selected_by_cv].set_index("city")
base = metrics[metrics.model == "dummy"].set_index("city")
stack = metrics[metrics.model == "stack"].set_index("city")
cmp = pd.DataFrame({"selected_model": sel["model"], "R2 selected": sel["r2_log"], "R2 stack": stack["r2_log"],
                    "R2 median-baseline": base["r2_log"], "WAPE selected": sel["wape"]})
display(cmp.round(3))
print("Mean test R2 (log): selected = %.3f | stack = %.3f | baseline = %.3f" %
      (cmp["R2 selected"].mean(), cmp["R2 stack"].mean(), cmp["R2 median-baseline"].mean()))

In [ ]:
pred = pd.read_parquet(run.eval / f"{CITIES[0]}_predictions.parquet")
best_label = metrics[(metrics.city == CITIES[0]) & metrics.selected_by_cv]["model"].iloc[0]
viz.plot_pred_vs_true(pred, best_label, CITIES[0]); plt.show()

## 5. What drives revenue? Permutation importance and cross-city agreement
Permutation importance is computed on **raw input columns** of the held-out hosts (an amenity list counts as one
feature), and city agreement is the Spearman correlation of the **full aligned importance vectors**.

In [ ]:
imps = {c: pd.read_csv(run.importance / f"{c}.csv") for c in CITIES}
viz.plot_importance(imps, cfg["evaluation"]["top_k_features"]); plt.show()
display(pd.DataFrame({c: d.head(5)["feature"].tolist() for c, d in imps.items()}, index=range(1, 6)))
if len(imps) > 1:
    viz.plot_rank_corr(rep["rank_corr"]); plt.show()

## 6. Reproducibility evidence

In [ ]:
manifest = json.loads((run.root / "manifest.json").read_text())
display(pd.Series({k: v for k, v in manifest.items() if k != "packages"}, name="manifest").to_frame())
display(pd.Series(manifest["packages"], name="package versions").to_frame().T)

In [ ]:
# Refit the CV-selected model of one city from scratch twice -> identical predictions.
c = CITIES[0]
res = {f"{r['model']}__{r['sampler']}": r for r in P.load_tuning(run, c)}
label = metrics[(metrics.city == c) & metrics.selected_by_cv]["model"].iloc[0]
X_tr, y_tr, g_tr, X_te, y_te, g_te, _ = P.load_split(run, c)
p1 = P.fit_tuned(run, c, res[label], X_tr, y_tr).predict(X_te)
p2 = P.fit_tuned(run, c, res[label], X_tr, y_tr).predict(X_te)
print(f"{c}/{label}: max |difference| between two independent refits = {np.abs(p1 - p2).max():.3g}")
assert np.allclose(p1, p2)
stored = pd.read_parquet(run.eval / f"{c}_predictions.parquet")[label].to_numpy()
print("matches stored evaluation predictions:", np.allclose(stored, p1))

## 7. Conclusions and honest caveats

* The numbers above are **out-of-sample on held-out hosts** and come from a pipeline that cannot see test data
  during preprocessing, tuning or stacking. v1's 88-99 % "accuracy" for Stack/Bayes/GA was an artefact of scoring on training data;
  the comparable v1 single-model figures (R² ≈ 0.6) are the honest ones, and v2's are not directly comparable because the target was redefined.
* Compare each model with the **median baseline and tuned Ridge**: a complex model is only interesting if the paired Wilcoxon
  test (section 4b) and the bootstrap CIs show a consistent gain.
* `host_*` features may rank high because professional multi-listing hosts price and operate differently – this is
  association, not causal advice.
* This is **estimated revenue (not profit)** for **listings active in the last 12 months**, from one scrape (Sept 2022);
  the occupancy heuristic should be stress-tested by varying `config.target` and re-running under a new `project.run_name`.
* If the notebook printed the *synthetic* banner in section 0, nothing here says anything about the Italian market.